In [2]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [6]:
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import pandas as pd
from src.models import create_custom_cnn


DATASET2_DIR = (
    "E:/Work/AI/MAKTAB/HW-CW-S02/"
    "FINAL BOSS/dataset/CROP+MERGED/train"
)

MODEL_DIR = (
    "E:/Work/AI/MAKTAB/HW-CW-S02/"
    "Vehicle Classification/Vehicle-Classification-Project/"
    "models/dataset2"
)

MODEL_PATHS = {
    "CNN baseline":
        f"{MODEL_DIR}/customcnn_d2_best.pth",

    "Balanced batches":
        f"{MODEL_DIR}/customcnn_d2_balancedbatches_d2_best.pth",

    "BCE":
        f"{MODEL_DIR}/customcnn_d2_bce_scheduler.pth",

    "CrossEntropy":
        f"{MODEL_DIR}/customcnn_d2_cross_entropy_scheduler.pth",

    "Scheduler":
        f"{MODEL_DIR}/customcnn_d2_scheduler.pth",

    "Weight Decay":
        f"{MODEL_DIR}/customcnn_d2_wd_1e-4.pth"
}


device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor()
])


dataset = datasets.ImageFolder(
    DATASET2_DIR,
    transform=transform
)

indices = np.arange(len(dataset))

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=dataset.targets
)

val_dataset = Subset(
    dataset,
    val_indices
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    pin_memory=True
)


def evaluate(model):

    model.eval()

    predictions = []
    targets = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)

            outputs = model(images)

            pred = outputs.argmax(dim=1)

            predictions.extend(
                pred.cpu().numpy()
            )

            targets.extend(
                labels.numpy()
            )

    precision = precision_score(
        targets,
        predictions,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        targets,
        predictions,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        targets,
        predictions,
        average="macro",
        zero_division=0
    )

    class_precision = precision_score(
        targets,
        predictions,
        average=None,
        zero_division=0
    )

    class_recall = recall_score(
        targets,
        predictions,
        average=None,
        zero_division=0
    )

    lowest_recall = dataset.classes[
        np.argmin(class_recall)
    ]

    lowest_precision = dataset.classes[
        np.argmin(class_precision)
    ]

    return (
        precision,
        recall,
        f1,
        lowest_recall,
        lowest_precision
    )


results = []


for name, path in MODEL_PATHS.items():

    model = create_custom_cnn(
        num_classes=8
    )

    model.load_state_dict(
        torch.load(
            path,
            map_location=device
        )
    )

    model = model.to(device)

    result = evaluate(model)

    results.append(
        [name, *result]
    )


In [7]:
df = pd.DataFrame(
    results,
    columns=[
        "Experiment",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Lowest-Recall Class",
        "Lowest-Precision Class"
    ]
)

df["Macro Precision"] *= 100
df["Macro Recall"] *= 100
df["Macro F1"] *= 100

df

,Experiment,Macro Precision,Macro Recall,Macro F1,Lowest-Recall Class,Lowest-Precision Class
0,CNN baseline,93.454455,93.256040,93.318776,kamyun,kamyunet
1,Balanced batches,92.315677,91.689997,91.887503,minibus,kamyun
2,BCE,92.167121,91.963601,92.004847,kamyun,kamyunet
3,CrossEntropy,88.281717,87.485815,87.795501,kamyunet,kamyun
4,Scheduler,93.094130,92.882804,92.919101,kamyun,kamyunet
5,Weight Decay,93.920713,93.751090,93.816818,kamyun,kamyun
